Investigating the CSB data set.


In [1]:
import duckdb
import pyogrio

In [2]:
GDB = "../in/CSB1825.gdb"  # relative to this notebook 

con = duckdb.connect()
con.sql("INSTALL spatial")
con.sql("LOAD spatial")

In [3]:
pyogrio.list_layers(GDB)

array([['national1825', 'MultiPolygon']], dtype=object)

In [9]:
con.sql(f"""
    DESCRIBE SELECT * FROM ST_Read('{GDB}', layer='national1825')
""")

┌──────────────┬───────────────────────┬─────────┬─────────┬─────────┬─────────┐
│ column_name  │      column_type      │  null   │   key   │ default │  extra  │
│   varchar    │        varchar        │ varchar │ varchar │ varchar │ varchar │
├──────────────┼───────────────────────┼─────────┼─────────┼─────────┼─────────┤
│ OBJECTID     │ BIGINT                │ YES     │ NULL    │ NULL    │ NULL    │
│ CSBID        │ VARCHAR               │ YES     │ NULL    │ NULL    │ NULL    │
│ CSBYEARS     │ VARCHAR               │ YES     │ NULL    │ NULL    │ NULL    │
│ CSBACRES     │ DOUBLE                │ YES     │ NULL    │ NULL    │ NULL    │
│ CDL2018      │ INTEGER               │ YES     │ NULL    │ NULL    │ NULL    │
│ CDL2019      │ INTEGER               │ YES     │ NULL    │ NULL    │ NULL    │
│ CDL2020      │ INTEGER               │ YES     │ NULL    │ NULL    │ NULL    │
│ CDL2021      │ INTEGER               │ YES     │ NULL    │ NULL    │ NULL    │
│ CDL2022      │ INTEGER    

In [6]:
con.sql(f"""
    SELECT *
    FROM ST_Read('{GDB}', layer='national1825')
    LIMIT 10
""")

┌──────────┬─────────────────┬──────────┬────────────────────┬─────────┬─────────┬─────────┬─────────┬─────────┬─────────┬─────────┬─────────┬───────────┬──────────┬─────────┬─────────┬──────────┬───────────────────┬──────────────┬──────────┬────────────────────┬────────────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [5]:
con.sql(f"""
    SELECT *, ST_AsText(Shape) AS shape_wkt
    FROM ST_Read('{GDB}', layer='national1825')
    LIMIT 10
""")

┌──────────┬─────────────────┬──────────┬────────────────────┬─────────┬─────────┬─────────┬─────────┬─────────┬─────────┬─────────┬─────────┬───────────┬──────────┬─────────┬─────────┬──────────┬───────────────────┬──────────────┬──────────┬────────────────────┬────────────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [7]:
con.sql(f"""
    SELECT count(*) AS n_features, count(DISTINCT STATEFIPS) AS n_states
    FROM ST_Read('{GDB}', layer='national1825')
""")

┌────────────┬──────────┐
│ n_features │ n_states │
│   int64    │  int64   │
├────────────┼──────────┤
│   15973095 │       48 │
└────────────┴──────────┘

In [8]:
info = pyogrio.read_info(GDB, layer="national1825")
info["crs"]

'PROJCS["Albers_Conic_Equal_Area",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]'

In [12]:
from pathlib import Path
import json

output_path = Path(GDB).resolve().parents[1] / "out" / "national1825.parquet"
output_path.parent.mkdir(parents=True, exist_ok=True)
if output_path.exists():
    raise FileExistsError(f"Output already exists: {output_path}")

rows_written = con.execute("""
    COPY (
        SELECT * FROM ST_Read($gdb, layer='national1825')
    ) TO $output (FORMAT PARQUET, COMPRESSION ZSTD)
""", {"gdb": GDB, "output": str(output_path)}).fetchone()[0]

metadata_row = con.execute("""
    SELECT decode(value) FROM parquet_kv_metadata(?)
    WHERE decode(key) = 'geo'
""", [str(output_path)]).fetchone()
assert metadata_row is not None, "GeoParquet metadata is missing"
geo_metadata = json.loads(metadata_row[0])
assert geo_metadata["columns"]["Shape"]["crs"]["id"]["code"] == 5070
rows_read = con.execute(
    "SELECT count(*) FROM read_parquet(?)", [str(output_path)]
).fetchone()[0]
assert rows_read == rows_written, "Export row count mismatch"
print(f"Wrote {rows_written:,} features to {output_path}")
print(f"GeoParquet {geo_metadata['version']}; CRS: EPSG:5070")

Wrote 15,973,095 features to /Users/jameshorine/repos/useful-views/crop-sequence-boundaries/out/national1825.parquet
GeoParquet 1.0.0; CRS: EPSG:5070
